# GOT-OCR-2.0 vs TrOCR: combined comparison

Runs both `stepfun-ai/GOT-OCR-2.0-hf` (from `GOT_testing.ipynb`) and
`microsoft/trocr-large-handwritten` (from `TROCR_testing.ipynb`) over every image in
`RiteshHandwritten/`, pretrained, no fine-tuning, same preprocessing as both source
notebooks.

Produces two reports:

1. **`ocr_comparison_report.csv`** -- per image: OCR text + confidence for GOT and for TrOCR.
2. **`ocr_accuracy_report.csv`** -- the same, joined against `Dataset_true.csv` (ground
   truth), with a text-similarity score (0-1, via `difflib`) for each model against the
   true label, plus each model's own confidence.

Paths below assume this notebook runs from the project root (`handwritten/`), with
`RiteshHandwritten/` and `Dataset_true.csv` alongside it. If running on Colab instead,
mount Drive and change `DIRECTORY_PATH` / `TRUE_CSV_PATH` accordingly.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled. Both models will be slow on CPU.")

In [ ]:
!pip install -q -U "transformers>=4.45" "tokenizers>=0.20" accelerate sentencepiece opencv-python-headless

In [ ]:
# Uncomment if running on Colab and the dataset lives on Drive instead of locally.
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os
import json
import difflib

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    AutoImageProcessor,
    RobertaTokenizer,
)

In [ ]:
DIRECTORY_PATH = "RiteshHandwritten"
TRUE_CSV_PATH = "Dataset_true.csv"
OUTPUT_DIR = "ocr_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

image_files = []
for root, _, files in os.walk(DIRECTORY_PATH):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_files.append(os.path.join(root, file))

image_files.sort(key=lambda p: os.path.basename(p))
print(f"Found {len(image_files)} image files in {DIRECTORY_PATH}")
if len(image_files) > 5:
    print("First 5 image files:", image_files[:5])
else:
    print("Image files:", image_files)

## Preprocessing

Same as both source notebooks: grayscale, upscale, local contrast boost, light denoise on
the whole page (no per-cell cropping).

In [ ]:
def preprocess_handwriting(image, scale=3):
    """
    Enhance a handwriting image for OCR: grayscale, upscale, local contrast boost,
    light denoise. Returns a grayscale numpy array.
    """
    img = np.array(image.convert("RGB"))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    upscaled = cv2.resize(
        gray,
        None,
        fx=scale,
        fy=scale,
        interpolation=cv2.INTER_CUBIC
    )

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )
    enhanced = clahe.apply(upscaled)

    denoised = cv2.fastNlMeansDenoising(
        enhanced,
        None,
        h=5,
        templateWindowSize=7,
        searchWindowSize=21
    )

    return denoised


PREPROCESS_SCALE = 3

## Line splitting (for TrOCR only)

GOT-OCR-2.0 reads the whole page directly. TrOCR reads one line at a time, so the page is
trimmed to its ink bounding box and split into lines via the horizontal ink profile, same
as `TROCR_testing.ipynb`.

In [ ]:
def ink_mask(gray):
    """Binary mask of dark ink on light paper (Otsu threshold)."""
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return mask


def trim_to_ink(gray, margin_ratio=0.15, min_ink_pixels=10):
    """
    Trim a grayscale image to the bounding box of its ink, then add a white margin.
    Returns None if the image has (almost) no ink.
    """
    if gray.size == 0:
        return None

    mask = ink_mask(gray)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))

    if np.count_nonzero(mask) < min_ink_pixels:
        return None

    ys, xs = np.nonzero(mask)
    y1, y2 = ys.min(), ys.max() + 1
    x1, x2 = xs.min(), xs.max() + 1
    trimmed = gray[y1:y2, x1:x2]

    m = int(min(trimmed.shape) * margin_ratio) + 4
    return cv2.copyMakeBorder(trimmed, m, m, m, m, cv2.BORDER_CONSTANT, value=255)


def split_into_lines(gray, min_line_ratio=0.25, min_gap_ratio=0.4):
    """
    Split a page crop into text-line images using the horizontal ink profile.
    Rows with (almost) no ink separate lines. Returns a list of grayscale arrays,
    top to bottom; returns [gray] when only one line is found.
    """
    mask = ink_mask(gray)
    profile = np.count_nonzero(mask, axis=1)

    if profile.max() == 0:
        return [gray]

    has_ink = profile > profile.max() * 0.05

    runs = []
    start = None
    for y, ink in enumerate(has_ink):
        if ink and start is None:
            start = y
        elif not ink and start is not None:
            runs.append([start, y])
            start = None
    if start is not None:
        runs.append([start, len(has_ink)])

    if not runs:
        return [gray]

    median_height = np.median([b - a for a, b in runs])
    min_gap = max(2, int(median_height * min_gap_ratio))
    merged = [runs[0]]
    for run in runs[1:]:
        if run[0] - merged[-1][1] < min_gap:
            merged[-1][1] = run[1]
        else:
            merged.append(run)

    tallest = max(b - a for a, b in merged)
    merged = [r for r in merged if (r[1] - r[0]) >= tallest * min_line_ratio]

    if len(merged) <= 1:
        return [gray]

    pad = max(2, min_gap // 2)
    return [gray[max(0, a - pad):min(gray.shape[0], b + pad)] for a, b in merged]


def prepare_page_for_trocr(enhanced):
    """
    Full page preparation: trim to ink, split into lines, trim each line.
    Returns a list of RGB PIL line images (empty list if no ink found).
    """
    trimmed = trim_to_ink(enhanced)
    if trimmed is None:
        return []

    line_images = []
    for line in split_into_lines(trimmed):
        line = trim_to_ink(line)
        if line is not None:
            line_images.append(Image.fromarray(line).convert("RGB"))
    return line_images

## Load GOT-OCR-2.0

In [ ]:
GOT_MODEL_ID = "stepfun-ai/GOT-OCR-2.0-hf"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading GOT-OCR-2.0...")

got_processor = AutoProcessor.from_pretrained(GOT_MODEL_ID)
got_model = AutoModelForImageTextToText.from_pretrained(
    GOT_MODEL_ID,
    device_map="auto",
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

print("GOT-OCR-2.0 loaded")

In [ ]:
def got_ocr(image, max_new_tokens=512, crop_to_patches=False, max_patches=3):
    """
    Run GOT-OCR 2.0 on an image (grayscale numpy array or PIL image) and return
    (decoded_text, confidence). Confidence is the mean per-token generation
    probability (exp of mean log-prob of the chosen token at each step).
    """
    if isinstance(image, np.ndarray):
        if len(image.shape) == 2:
            image = Image.fromarray(image)
        else:
            image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    image = image.convert("RGB")

    processor_kwargs = {"return_tensors": "pt"}

    if crop_to_patches:
        processor_kwargs["crop_to_patches"] = True
        processor_kwargs["max_patches"] = max_patches

    inputs = got_processor(image, **processor_kwargs)

    inputs = {
        k: v.to(got_model.device) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        outputs = got_model.generate(
            **inputs,
            do_sample=False,
            tokenizer=got_processor.tokenizer,
            stop_strings="<|im_end|>",
            max_new_tokens=max_new_tokens,
            output_scores=True,
            return_dict_in_generate=True,
        )

    generated_ids = outputs.sequences
    input_length = inputs["input_ids"].shape[1]
    new_token_ids = generated_ids[0][input_length:]

    generated_text = got_processor.decode(
        new_token_ids,
        skip_special_tokens=True
    )

    token_probs = []
    for step_scores, token_id in zip(outputs.scores, new_token_ids):
        if token_id in got_processor.tokenizer.all_special_ids:
            continue
        step_probs = torch.softmax(step_scores[0], dim=-1)
        token_probs.append(step_probs[token_id].item())

    confidence = float(np.mean(token_probs)) if token_probs else float("nan")

    return generated_text.strip(), confidence


def got_read_image(image, scale=PREPROCESS_SCALE, max_new_tokens=512, crop_to_patches=False, max_patches=3):
    """
    Preprocess + OCR one page image with GOT-OCR-2.0 (no cell/grid/line splitting).
    Returns (text, confidence).
    """
    enhanced = preprocess_handwriting(image, scale=scale)
    text, confidence = got_ocr(
        enhanced,
        max_new_tokens=max_new_tokens,
        crop_to_patches=crop_to_patches,
        max_patches=max_patches
    )
    return text, confidence

## Load TrOCR (`microsoft/trocr-large-handwritten`)

Forces the slow (BPE) `RobertaTokenizer` -- the fast-tokenizer conversion path can raise
"Couldn't instantiate the backend tokenizer" on this repo even with `sentencepiece`
installed.

In [ ]:
TROCR_MODEL_ID = "microsoft/trocr-large-handwritten"

print("Loading TrOCR...")

trocr_image_processor = AutoImageProcessor.from_pretrained(TROCR_MODEL_ID)
trocr_tokenizer = RobertaTokenizer.from_pretrained(TROCR_MODEL_ID, use_fast=False)
trocr_processor = TrOCRProcessor(image_processor=trocr_image_processor, tokenizer=trocr_tokenizer)

trocr_model = VisionEncoderDecoderModel.from_pretrained(
    TROCR_MODEL_ID,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to(device)
trocr_model.eval()

print("TrOCR loaded on", device)

In [ ]:
def trocr_ocr_batch(line_images, batch_size=16, num_beams=4, max_new_tokens=64):
    """
    Run TrOCR on a list of single-line PIL images.
    Returns (texts, confidences) -- confidences are per-line probabilities derived
    from the beam search's length-normalized sequence log-probability
    (`sequences_scores`).
    """
    texts = []
    confidences = []
    for i in range(0, len(line_images), batch_size):
        batch = line_images[i:i + batch_size]
        pixel_values = trocr_processor(images=batch, return_tensors="pt").pixel_values
        pixel_values = pixel_values.to(device, dtype=trocr_model.dtype)

        with torch.inference_mode():
            outputs = trocr_model.generate(
                pixel_values,
                num_beams=num_beams,
                max_new_tokens=max_new_tokens,
                early_stopping=True,
                output_scores=True,
                return_dict_in_generate=True,
            )

        texts.extend(
            t.strip() for t in trocr_processor.batch_decode(outputs.sequences, skip_special_tokens=True)
        )
        confidences.extend(
            torch.exp(outputs.sequences_scores).tolist()
        )
    return texts, confidences


def trocr_read_image(image, scale=PREPROCESS_SCALE):
    """
    Preprocess + line-split + OCR one image (no cell/grid detection).
    Returns (full_text, confidence). `confidence` is the mean of the per-line
    confidences (page-level proxy).
    """
    enhanced = preprocess_handwriting(image, scale=scale)
    line_images = prepare_page_for_trocr(enhanced)
    if line_images:
        line_texts, line_confidences = trocr_ocr_batch(line_images)
    else:
        line_texts, line_confidences = [], []
    full_text = "\n".join(line_texts)
    confidence = float(np.mean(line_confidences)) if line_confidences else float("nan")
    return full_text, confidence

## Try both models on one image

In [ ]:
preview_image = Image.open(image_files[0]).convert("RGB")

got_preview_text, got_preview_conf = got_read_image(preview_image)
trocr_preview_text, trocr_preview_conf = trocr_read_image(preview_image)

print(os.path.basename(image_files[0]))
print("\n--- GOT-OCR-2.0 ---")
print(f"Confidence: {got_preview_conf:.4f}")
print(got_preview_text)
print("\n--- TrOCR ---")
print(f"Confidence: {trocr_preview_conf:.4f}")
print(trocr_preview_text)

plt.figure(figsize=(10, 5))
plt.imshow(preview_image)
plt.axis("off")
plt.title(os.path.basename(image_files[0]))
plt.show()

## Run both models on every image

In [ ]:
results = {}

for img_path in image_files:
    name = os.path.basename(img_path)
    print(f"Processing image: {name}")
    image = Image.open(img_path).convert("RGB")

    try:
        got_text, got_confidence = got_read_image(image)
    except Exception as e:
        print(f"  GOT-OCR-2.0 error on {name}: {e}")
        got_text, got_confidence = "", float("nan")

    try:
        trocr_text, trocr_confidence = trocr_read_image(image)
    except Exception as e:
        print(f"  TrOCR error on {name}: {e}")
        trocr_text, trocr_confidence = "", float("nan")

    results[name] = {
        "got_text": got_text,
        "got_confidence": got_confidence,
        "trocr_text": trocr_text,
        "trocr_confidence": trocr_confidence,
    }
    print(f"  -> GOT confidence: {got_confidence:.4f} | TrOCR confidence: {trocr_confidence:.4f}")

print(f"\nProcessed {len(results)} image(s).")

## Report 1: OCR text + confidence per image, both models

In [ ]:
comparison_rows = [
    {
        "image_name": name,
        "got_text": r["got_text"],
        "got_confidence": r["got_confidence"],
        "trocr_text": r["trocr_text"],
        "trocr_confidence": r["trocr_confidence"],
    }
    for name, r in results.items()
]

comparison_df = pd.DataFrame(comparison_rows).sort_values("image_name").reset_index(drop=True)

comparison_csv_path = os.path.join(OUTPUT_DIR, "ocr_comparison_report.csv")
comparison_df.to_csv(comparison_csv_path, index=False, encoding="utf-8")

with open(os.path.join(OUTPUT_DIR, "ocr_comparison_report.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Saved:", comparison_csv_path)
comparison_df

## Report 2: compare against ground truth (`Dataset_true.csv`)

Similarity is `difflib.SequenceMatcher` ratio (0-1) between the model's OCR text and the
true label, case-insensitive and whitespace-normalized -- a simple stand-in for accuracy
against handwritten free-text (exact match is too strict for OCR).

`Dataset_true.csv` uses the literal string `N/A` where the ground truth for an image is
unknown/illegible, not as the string the model is expected to output. Those rows get
`ground_truth_known = False` and a blank similarity score (`NaN`), and are excluded from
the summary averages below -- otherwise they'd unfairly reward or penalize a model for
matching/missing a placeholder that isn't a real label.

In [ ]:
def normalize_text(text):
    return " ".join(str(text).strip().lower().split())


def similarity(a, b):
    a, b = normalize_text(a), normalize_text(b)
    if not a and not b:
        return 1.0
    return difflib.SequenceMatcher(None, a, b).ratio()


true_df = pd.read_csv(TRUE_CSV_PATH)
true_df["image_name"] = true_df["image_name"].astype(str).str.strip()

# comparison_df image names include extensions (e.g. Test_img1.png); true_df doesn't.
comparison_df["image_stem"] = comparison_df["image_name"].apply(lambda n: os.path.splitext(n)[0])

accuracy_df = comparison_df.merge(
    true_df, left_on="image_stem", right_on="image_name", how="left", suffixes=("", "_true")
).drop(columns=["image_stem", "image_name_true"])

normalized_true = accuracy_df["true_text"].apply(normalize_text)
accuracy_df["ground_truth_known"] = ~normalized_true.isin(["", "n/a"])

accuracy_df["got_similarity"] = accuracy_df.apply(
    lambda row: similarity(row["got_text"], row["true_text"]) if row["ground_truth_known"] else np.nan,
    axis=1,
)
accuracy_df["trocr_similarity"] = accuracy_df.apply(
    lambda row: similarity(row["trocr_text"], row["true_text"]) if row["ground_truth_known"] else np.nan,
    axis=1,
)

accuracy_df = accuracy_df[
    [
        "image_name", "true_text", "ground_truth_known",
        "got_text", "got_confidence", "got_similarity",
        "trocr_text", "trocr_confidence", "trocr_similarity",
    ]
].sort_values("image_name").reset_index(drop=True)

accuracy_csv_path = os.path.join(OUTPUT_DIR, "ocr_accuracy_report.csv")
accuracy_df.to_csv(accuracy_csv_path, index=False, encoding="utf-8")

print("Saved:", accuracy_csv_path)
print(f"Ground truth unknown (N/A) for {(~accuracy_df['ground_truth_known']).sum()} of {len(accuracy_df)} image(s).")
accuracy_df

In [ ]:
## Append the image path to ocr_accuracy_report.csv

image_path_by_name = {os.path.basename(p): p for p in image_files}
accuracy_df["image_path"] = accuracy_df["image_name"].map(image_path_by_name)

accuracy_df.to_csv(accuracy_csv_path, index=False, encoding="utf-8")

print("Updated:", accuracy_csv_path, "with image_path column")
accuracy_df[["image_name", "image_path"]].head()

In [ ]:
known_df = accuracy_df[accuracy_df["ground_truth_known"]]

summary = pd.DataFrame({
    "model": ["GOT-OCR-2.0", "TrOCR"],
    "mean_confidence": [accuracy_df["got_confidence"].mean(), accuracy_df["trocr_confidence"].mean()],
    "mean_similarity_to_truth": [known_df["got_similarity"].mean(), known_df["trocr_similarity"].mean()],
    "exact_match_rate": [
        (known_df["got_similarity"] == 1.0).mean(),
        (known_df["trocr_similarity"] == 1.0).mean(),
    ],
    "images_evaluated": [len(known_df), len(known_df)],
})

summary_csv_path = os.path.join(OUTPUT_DIR, "ocr_accuracy_summary.csv")
summary.to_csv(summary_csv_path, index=False, encoding="utf-8")

print("Saved:", summary_csv_path)
print(f"(excluded {len(accuracy_df) - len(known_df)} image(s) with unknown ground truth)")
summary